In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.iolib.summary2 import summary_col

# Не показывать Warning
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Настройки оформления графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

CONST_NAMES = ('const', 'Intercept')


def fmt(x, decimals=3):
    """Число с заданным числом знаков; оценка, которая округляется до нуля, — с тремя значащими цифрами."""
    if pd.isna(x):
        return ''
    text = f'{x:.{decimals}f}'
    if float(text) == 0:
        return f'{x:.3g}' if x != 0 else f'{0:.{decimals}f}'
    return text


def fmt_p(p, decimals=3):
    """P-значение: меньше 0.001 печатается как «<0.001»."""
    return '<0.001' if p < 0.001 else f'{p:.{decimals}f}'


def errors_text(res, case='nom'):
    """Название типа стандартных ошибок подгонки: 'nom' — «неробастные», 'dat' — «по неробастным …»."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    if cov_type == 'nonrobust':
        return {'nom': 'неробастные стандартные ошибки',
                'dat': 'по неробастным стандартным ошибкам'}[case]
    return {'nom': f'робастные стандартные ошибки, {cov_type}',
            'dat': f'по робастным стандартным ошибкам {cov_type}'}[case]


def fit_ols(formula, data, robust=False, **kwargs):
    """МНК-подгонка; robust=True — робастные ошибки HC3. Всегда t- и F-распределения (use_t=True)."""
    cov_type = 'HC3' if robust else 'nonrobust'
    return ols(formula, data=data).fit(cov_type=cov_type, use_t=True, **kwargs)


def show_table(df, index=True, align=None):
    """Таблица pandas в Markdown; числа должны быть заранее превращены в строки через fmt."""
    display(Markdown('\n\n' + df.to_markdown(index=index, colalign=align, disable_numparse=True) + '\n\n'))


def nobs_text(res):
    """Строка «Модель была подогнана по N наблюдениям»."""
    n = int(res.nobs)
    word = 'наблюдению' if n % 10 == 1 and n % 100 != 11 else 'наблюдениям'
    return f'Модель была подогнана по {n} {word}.'


def show_coefs(res, decimals=3, show_nobs=True):
    """Оценки коэффициентов: вертикальная таблица «Переменная | Оценка» и число наблюдений."""
    cov_type = getattr(res, 'cov_type', 'nonrobust')
    column = 'Оценка' if cov_type == 'nonrobust' else f'Оценка ({cov_type})'
    table = pd.DataFrame({'Переменная': res.params.index,
                          column: [fmt(b, decimals) for b in res.params]})
    text = table.to_markdown(index=False, colalign=('left', 'right'), disable_numparse=True)
    if show_nobs:
        text += '\n\n' + nobs_text(res)
    display(Markdown('\n\n' + text + '\n\n'))


def display_answer(text, use_callout=True, theme='note', title='Критическое значение'):
    """Ответ в рамке (callout): theme='note' — число, 'tip' — вывод, 'important' — условие."""
    if use_callout:
        head = f'title="{title}" ' if title else ''
        output_md = f'\n\n::: {{.callout-{theme} {head}icon="true"}}\n{text}\n:::\n\n'
    elif title:
        output_md = f'\n\n**{title}:** {text}\n\n'
    else:
        output_md = f'\n\n{text}\n\n'
    display(Markdown(output_md))


def show_conditions(level, errors=None, hypothesis=None, extra=()):
    """Блок «Условия теста»: уровень значимости (доля или список долей), тип ошибок, гипотеза формулой."""
    levels = level if isinstance(level, (list, tuple)) else [level]
    items = ['уровень значимости: **' + ' и '.join(f'{a * 100:g}%' for a in levels) + '**']
    if errors:
        items.append(f'стандартные ошибки: **{errors}**')
    hypotheses = [hypothesis] if isinstance(hypothesis, str) else list(hypothesis or [])
    if len(hypotheses) == 1:
        items.append(f'гипотеза: {hypotheses[0]}')
    else:
        items += [f'гипотеза {i}: {h}' for i, h in enumerate(hypotheses, start=1)]
    items += list(extra)
    display_answer('\n'.join(f'- {item}' for item in items), theme='important', title='Условия теста')


def signif_stars(p):
    """Коды значимости: *** p<0.01, ** p<0.05, * p<0.1."""
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''


SIGNIF_LEGEND = '*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$'


def print_regression_summary(res, decimals=3, show_signif=True):
    """Таблица t-теста: Coef., Std.Err., t-value, p-value, Signif. Заголовок называет тип ошибок."""
    title = f'**Результаты $t$-теста для коэффициентов ({errors_text(res)}):**'
    table = pd.DataFrame({
        'Coef.': [fmt(b, decimals) for b in res.params],
        'Std.Err.': [fmt(s, decimals) for s in res.bse],
        't-value': [fmt(t, decimals) for t in res.tvalues],
        'p-value': [fmt_p(p, decimals) for p in res.pvalues],
    }, index=res.params.index)
    legend = ''
    if show_signif:
        table['Signif.'] = ['' if name in CONST_NAMES else signif_stars(p)
                            for name, p in res.pvalues.items()]
        legend = '\n\n' + SIGNIF_LEGEND
    align = ('left',) + ('right',) * 4 + (('left',) if show_signif else ())
    display(Markdown(f'\n\n{title}\n\n' + table.to_markdown(colalign=align, disable_numparse=True) + legend + '\n\n'))


def print_significant_coeffs(res, sign_level=0.05, theme='note', title=None):
    """Рамка со списком коэффициентов, значимых на уровне sign_level (без константы)."""
    names = [name for name, p in res.pvalues.items() if p < sign_level and name not in CONST_NAMES]
    if title is None:
        title = f'Коэффициенты, значимые на уровне {sign_level * 100:g}% ({errors_text(res, "dat")})'
    text = ', '.join(f'`{name}`' for name in names) if names else 'Значимых коэффициентов не выявлено.'
    display_answer(text, theme=theme, title=title)


def show_models(models, names=None, decimals=3, regressor_order=(), errors_note=None, info=None):
    """Сравнение регрессий (summary_col): строка Dep. Variable, коды значимости, пояснение про скобки.

    info — словарь {название строки: функция от подгонки} для дополнительных строк (N, F и т. п.).
    """
    table = summary_col(models, model_names=names, stars=True, float_format=f'%0.{decimals}f',
                        regressor_order=list(regressor_order), info_dict=info or {}).tables[0]
    top = pd.DataFrame({col: [m.model.endog_names] for col, m in zip(table.columns, models)},
                       index=['Dep. Variable'])
    table = pd.concat([top, table])
    if errors_note is None:
        errors_note = errors_text(models[0])
    note = f'{SIGNIF_LEGEND}. В скобках — {errors_note}.'
    display(Markdown('\n\n' + table.to_markdown(disable_numparse=True) + '\n\n' + note + '\n\n'))


def show_ci(res, level=0.95, decimals=3):
    """Доверительные интервалы: «Переменная | Нижняя граница | Верхняя граница», заголовок над таблицей."""
    ci = res.conf_int(alpha=1 - level)
    table = pd.DataFrame({'Переменная': ci.index,
                          'Нижняя граница': [fmt(x, decimals) for x in ci[0]],
                          'Верхняя граница': [fmt(x, decimals) for x in ci[1]]})
    title = f'**{level * 100:g}%-е доверительные интервалы ({errors_text(res)}):**'
    display(Markdown(f'\n\n{title}\n\n'
                     + table.to_markdown(index=False, colalign=('left', 'right', 'right'), disable_numparse=True) + '\n\n'))


# Загрузка данных
sleep75 = pd.read_csv('../datasets/sleep75.csv')
wage1 = pd.read_csv('../datasets/wage1.csv')
wage2 = pd.read_csv('../datasets/wage2.csv')
Labour = pd.read_csv('../datasets/Labour.csv')

# Помощники листка
my_digits = 2


def show_text_table(table, align):
    """Таблица из готовых строк: общий show_table даёт tabulate переразобрать числа («6.800» → «6.8»)."""
    text = table.to_markdown(index=False, colalign=align, disable_numparse=True)
    display(Markdown('\n\n' + text + '\n\n'))


def show_coefs_signif(res, digits=4):
    """Оценки с заданным числом значащих цифр (общий show_coefs печатает фиксированное число знаков)."""
    table = pd.DataFrame({'Переменная': res.params.index,
                          'Оценка': [f'{b:.{digits}g}' for b in res.params]})
    show_text_table(table, ('left', 'right'))
    display(Markdown(nobs_text(res) + '\n\n'))


def show_new(new_df, unit='Человек'):
    """Таблица характеристик новых наблюдений с номерами 1, 2, 3."""
    table = new_df.copy()
    table.insert(0, unit, range(1, len(table) + 1))
    # Строка таблицы длиннее 72 знаков растягивает таблицу в pdf на всю ширину — убираем выравнивание пробелами.
    text = table.to_markdown(index=False).replace('  ', ' ').replace('  ', ' ').replace('--', '-')
    display(Markdown('\n\n' + text + '\n\n'))


def show_predict(res, new_df, title, unit='Человек', log=False):
    """Таблица прогнозов; log=True — рядом с прогнозом логарифма прогноз в уровнях (экспонента)."""
    pred = res.predict(new_df)
    table = pd.DataFrame({unit: range(1, len(pred) + 1)})
    if log:
        log_name = res.model.endog_names
        level_name = log_name[len('np.log('):-1]
        table[f'Прогноз {log_name}'] = [fmt(x, 3) for x in pred]
        table[f'Прогноз {level_name}'] = [fmt(x, my_digits) for x in np.exp(pred)]
    else:
        table['Прогноз'] = [fmt(x, my_digits) for x in pred]
    display(Markdown(f'\n\n**{title}:**\n\n'))
    show_text_table(table, ('right',) * table.shape[1])

::: {.callout-important title="Важно"}
- Логарифм натуральный; константа включается в модель по умолчанию.
- Данные лежат в папке `datasets`, описание переменных — в `datasets/data-description.md`.
- Ответы округляются до 2-х десятичных знаков.
- Коэффициенты подписаны так, как их печатает statsmodels: `Intercept` — константа, `np.log(x)` — $\log x$, `I(x ** 2)` — $x^2$.
- Прогноз считается по неокруглённым оценкам (метод `predict`); расчёт по округлённым числам из таблицы может дать другой ответ.
- В логарифмических моделях прогноз в уровнях — экспонента от прогноза логарифма (без поправки).
:::

# Прогноз в модели в уровнях

## sleep equation #1

Для набора данных `sleep75` (`sleep` — время ночного сна, `totwrk` — время работы, обе в минутах в неделю) рассмотрим регрессию

`sleep ~ totwrk + age + south + male + smsa + yngkid + marr`

Результаты оценивания:

In [2]:
#| output: asis
regr = fit_ols('sleep ~ totwrk + age + south + male + smsa + yngkid + marr', sleep75)
show_coefs_signif(regr)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |     3451 |
| totwrk       |  -0.1692 |
| age          |    2.689 |
| south        |    101.6 |
| male         |    87.67 |
| smsa         |   -54.75 |
| yngkid       |   -13.96 |
| marr         |    31.21 |



Модель была подогнана по 706 наблюдениям.



Рассмотрим трёх людей с характеристиками

In [3]:
#| output: asis
new_df = pd.DataFrame({'totwrk': [2150, 1950, 2240], 'age': [37, 28, 26],
                       'south': [0, 1, 0], 'male': [1, 1, 0], 'smsa': [1, 0, 1],
                       'yngkid': [0, 1, 0], 'marr': [1, 0, 0]})
show_new(new_df)



| Человек | totwrk | age | south | male | smsa | yngkid | marr |
|-----:|-----:|---:|----:|----:|----:|-----:|----:|
|   1 |  2150 | 37 |  0 |  1 |  1 |  0 |  1 |
|   2 |  1950 | 28 |  1 |  1 |  0 |  1 |  0 |
|   3 |  2240 | 26 |  0 |  0 |  1 |  0 |  0 |



Вычислите прогноз для каждого человека по подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [4]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы sleep (минут в неделю)')



**Прогнозы sleep (минут в неделю):**





|   Человек |   Прогноз |
|----------:|----------:|
|         1 |   3250.68 |
|         2 |   3371.46 |
|         3 |   3086.98 |



## sleep equation #2

Для набора данных `sleep75` рассмотрим регрессию

`sleep ~ totwrk + age + I(age**2) + south + male`

Результаты оценивания:

In [5]:
#| output: asis
regr = fit_ols('sleep ~ totwrk + age + I(age**2) + south + male', sleep75)
show_coefs_signif(regr)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |     3614 |
| totwrk       |  -0.1673 |
| age          |   -6.255 |
| I(age ** 2)  |   0.1089 |
| south        |    114.5 |
| male         |    90.46 |



Модель была подогнана по 706 наблюдениям.



Рассмотрим трёх людей с характеристиками

In [6]:
#| output: asis
new_df = pd.DataFrame({'totwrk': [2160, 1720, 2390], 'age': [32, 24, 44],
                       'south': [1, 0, 0], 'male': [0, 1, 1]})
show_new(new_df)



| Человек | totwrk | age | south | male |
|-----:|-----:|---:|----:|----:|
|   1 |  2160 | 32 |  1 |  0 |
|   2 |  1720 | 24 |  0 |  1 |
|   3 |  2390 | 44 |  0 |  1 |



Вычислите прогноз для каждого человека по подогнанной модели. **Ответ округлите до 2-х десятичных знаков.**

In [7]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы sleep (минут в неделю)')



**Прогнозы sleep (минут в неделю):**





|   Человек |   Прогноз |
|----------:|----------:|
|         1 |   3278.46 |
|         2 |   3329.25 |
|         3 |   3240.11 |



# Прогноз в логарифмической модели

## wage equation #1

Для набора данных `wage2` (`wage` — месячный заработок) рассмотрим регрессию

`np.log(wage) ~ age + IQ + south + married + urban`

Результаты оценивания:

In [8]:
#| output: asis
regr = fit_ols('np.log(wage) ~ age + IQ + south + married + urban', wage2)
show_coefs_signif(regr)



| Переменная   |   Оценка |
|:-------------|---------:|
| Intercept    |    4.974 |
| age          |  0.02125 |
| IQ           | 0.008201 |
| south        | -0.09898 |
| married      |    0.201 |
| urban        |    0.175 |



Модель была подогнана по 935 наблюдениям.



Рассмотрим трёх людей с характеристиками

In [9]:
#| output: asis
new_df = pd.DataFrame({'age': [36, 29, 25], 'IQ': [105, 123, 112],
                       'south': [1, 0, 1], 'married': [1, 1, 0], 'urban': [1, 0, 1]})
show_new(new_df)



| Человек | age | IQ | south | married | urban |
|-----:|---:|---:|----:|-----:|----:|
|   1 | 36 | 105 |  1 |   1 |  1 |
|   2 | 29 | 123 |  0 |   1 |  0 |
|   3 | 25 | 112 |  1 |   0 |  1 |



Вычислите прогноз **зарплаты** для каждого человека по подогнанной модели. Прогноз в уровнях получите как экспоненту от прогноза логарифма (без поправки). **Ответ округлите до 2-х десятичных знаков.**

In [10]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы wage (месячный заработок)', log=True)



**Прогнозы wage (месячный заработок):**





|   Человек |   Прогноз np.log(wage) |   Прогноз wage |
|----------:|-----------------------:|---------------:|
|         1 |                  6.877 |         969.95 |
|         2 |                  6.800 |         897.93 |
|         3 |                  6.500 |         665.07 |



## wage equation #2

Для набора данных `wage1` (`wage` — средний почасовой заработок) рассмотрим регрессию

`np.log(wage) ~ exper + I(exper**2) + female + married + smsa`

Результаты оценивания:

In [11]:
#| output: asis
regr = fit_ols('np.log(wage) ~ exper + I(exper**2) + female + married + smsa', wage1)
show_coefs_signif(regr)



| Переменная    |     Оценка |
|:--------------|-----------:|
| Intercept     |       1.25 |
| exper         |    0.03688 |
| I(exper ** 2) | -0.0007717 |
| female        |    -0.3627 |
| married       |     0.1438 |
| smsa          |     0.2726 |



Модель была подогнана по 526 наблюдениям.



Рассмотрим трёх людей с характеристиками

In [12]:
#| output: asis
new_df = pd.DataFrame({'exper': [5, 26, 38], 'female': [1, 0, 1],
                       'married': [1, 0, 1], 'smsa': [1, 1, 0]})
show_new(new_df)



| Человек | exper | female | married | smsa |
|-----:|----:|-----:|-----:|----:|
|   1 |  5 |  1 |   1 |  1 |
|   2 |  26 |  0 |   0 |  1 |
|   3 |  38 |  1 |   1 |  0 |



Вычислите прогноз **зарплаты** для каждого человека по подогнанной модели. Прогноз в уровнях получите как экспоненту от прогноза логарифма (без поправки). **Ответ округлите до 2-х десятичных знаков.**

In [13]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы wage (средний почасовой заработок)', log=True)



**Прогнозы wage (средний почасовой заработок):**





|   Человек |   Прогноз np.log(wage) |   Прогноз wage |
|----------:|-----------------------:|---------------:|
|         1 |                  1.469 |           4.35 |
|         2 |                  1.960 |           7.10 |
|         3 |                  1.319 |           3.74 |



## output equation #1

Для набора данных `Labour` (`output` — добавленная стоимость, `capital` — основной капитал, обе в млн евро; `labour` — число работников) рассмотрим регрессию **np.log(output) на np.log(capital), np.log(labour)**.

Результаты оценивания:

In [14]:
#| output: asis
regr = fit_ols('np.log(output) ~ np.log(capital) + np.log(labour)', Labour)
show_coefs_signif(regr)



| Переменная      |   Оценка |
|:----------------|---------:|
| Intercept       |   -1.711 |
| np.log(capital) |   0.2076 |
| np.log(labour)  |   0.7148 |



Модель была подогнана по 569 наблюдениям.



Рассмотрим три фирмы с характеристиками

In [15]:
#| output: asis
new_df = pd.DataFrame({'capital': [2.97, 10.45, 3.85], 'labour': [85, 60, 105]})
show_new(new_df, unit='Фирма')



| Фирма | capital | labour |
|----:|-----:|-----:|
|  1 |  2.97 |  85 |
|  2 |  10.45 |  60 |
|  3 |  3.85 |  105 |



Вычислите прогноз **выпуска** для каждой фирмы по подогнанной модели. Прогноз в уровнях получите как экспоненту от прогноза логарифма (без поправки). **Ответ округлите до 2-х десятичных знаков.**

In [16]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы output (млн евро)', unit='Фирма', log=True)



**Прогнозы output (млн евро):**





|   Фирма |   Прогноз np.log(output) |   Прогноз output |
|--------:|-------------------------:|-----------------:|
|       1 |                    1.690 |             5.42 |
|       2 |                    1.702 |             5.49 |
|       3 |                    1.895 |             6.65 |



## output equation #2

Для набора данных `Labour` рассмотрим регрессию

```
np.log(output) ~ np.log(capital) + np.log(labour) +
    I(np.log(capital)**2) + I(np.log(labour)**2)
```

Результаты оценивания:

In [17]:
#| output: asis
regr = fit_ols('np.log(output) ~ np.log(capital) + np.log(labour)'
               ' + I(np.log(capital)**2) + I(np.log(labour)**2)', Labour)
show_coefs_signif(regr)



| Переменная              |   Оценка |
|:------------------------|---------:|
| Intercept               |   -1.304 |
| np.log(capital)         |   0.1831 |
| np.log(labour)          |   0.5153 |
| I(np.log(capital) ** 2) |  0.02275 |
| I(np.log(labour) ** 2)  |  0.02026 |



Модель была подогнана по 569 наблюдениям.



Рассмотрим три фирмы с характеристиками

In [18]:
#| output: asis
new_df = pd.DataFrame({'capital': [22.14, 7.32, 0.67], 'labour': [407, 197, 31]})
show_new(new_df, unit='Фирма')



| Фирма | capital | labour |
|----:|-----:|-----:|
|  1 |  22.14 |  407 |
|  2 |  7.32 |  197 |
|  3 |  0.67 |  31 |



Вычислите прогноз **выпуска** для каждой фирмы по подогнанной модели. Прогноз в уровнях получите как экспоненту от прогноза логарифма (без поправки). **Ответ округлите до 2-х десятичных знаков.**

In [19]:
#| output: asis
show_predict(regr, new_df, 'Прогнозы output (млн евро)', unit='Фирма', log=True)



**Прогнозы output (млн евро):**





|   Фирма |   Прогноз np.log(output) |   Прогноз output |
|--------:|-------------------------:|-----------------:|
|       1 |                    3.309 |            27.37 |
|       2 |                    2.439 |            11.46 |
|       3 |                    0.635 |             1.89 |

